In [ ]:
#https://www.kaggle.com/competitions/map-charting-student-math-misunderstandings

#https://www.kaggle.com/datasets/jaytonde/deekseepmath-7b-map-competition/versions/1
#https://www.kaggle.com/datasets/jjsdfd22/deepseek-prover-v2-7b-bf16
#https://www.kaggle.com/datasets/jjsdfd22/deepseek-prover-v2-7b-map
#https://www.kaggle.com/datasets/jjsdfd22/qwen2-5-math-7b-instruct-map/versions/1

#https://www.kaggle.com/datasets/ilyass323/qwen3-4b-full-map-competition/versions/1
#https://www.kaggle.com/datasets/jaytonde/qwen3-8b-map-competition
#https://www.kaggle.com/models/qwen-lm/qwen2.5-math/Transformers/7b-instruct/1
#https://www.kaggle.com/datasets/jjsdfd22/hunyuan-7b-instruct-bf16

#https://www.kaggle.com/datasets/jjsdfd22/hunyuan-7b-instruct-map
#https://www.kaggle.com/datasets/jjsdfd22/transformers-4-56-1-and-deps/versions/1




In [ ]:
#公开模型1-https://www.kaggle.com/datasets/ilyass323/qwen3-4b-full-map-competition
%%writefile qwen3_4b_inference.py
import os
import torch
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from scipy.special import softmax

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
train = pd.read_csv('/kaggle/input/map-charting-student-math-misunderstandings/train.csv')
test  = pd.read_csv('/kaggle/input/map-charting-student-math-misunderstandings/test.csv')
idx = train.apply(lambda row: row.Category.split('_')[0], axis=1) == 'True'
correct = train.loc[idx].copy()
correct['c'] = correct.groupby(['QuestionId', 'MC_Answer']).MC_Answer.transform('count')
correct = correct.sort_values('c', ascending=False).drop_duplicates(['QuestionId'])
correct = correct[['QuestionId', 'MC_Answer']]
correct['is_correct'] = 1

train = train.merge(correct, on=['QuestionId','MC_Answer'], how='left')
train.is_correct = train.is_correct.fillna(0)

test = test.merge(correct, on=['QuestionId','MC_Answer'], how='left')
test.is_correct = test.is_correct.fillna(0)

def format_input(row):
    x = "This answer is correct." if row['is_correct'] else "This answer is incorrect."
    return (
        f"Question: {row['QuestionText']}\n"
        f"Answer: {row['MC_Answer']}\n"
        f"{x}\n"
        f"Student Explanation: {row['StudentExplanation']}"
    )

test['text'] = test.apply(format_input, axis=1)

# --- Dataset ---
ds_test = Dataset.from_pandas(test[['row_id', 'text']])
model_name = "/kaggle/input/qwen3-4b-full-map-competition"
# --- Load label encoder mapping ---
with open(os.path.join(model_name, "label_encoder.json")) as f:
    label_data = json.load(f)
target_classes = label_data["classes"]
idx2label = {i: lbl for i, lbl in enumerate(target_classes)}
label2idx = {lbl: i for i, lbl in enumerate(target_classes)}

n_classes = len(target_classes)
print(f"Train shape: {train.shape} with {n_classes} target classes")
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=n_classes,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

model.config.pad_token_id = tokenizer.pad_token_id

ds_test = ds_test.map(tokenize, batched=True)

test_args = TrainingArguments(
    output_dir="./",
    do_train=False,
    do_predict=True,
    per_device_eval_batch_size=16,
    fp16=True,
    bf16=False,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=test_args,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer)
)

predictions = trainer.predict(ds_test)
probs = softmax(predictions.predictions, axis=1)

# --- Top-3 predictions ---
top3 = np.argsort(-probs, axis=1)[:, :25]
top3_labels = np.vectorize(idx2label.get)(top3)
joined_preds = [" ".join(row) for row in top3_labels]

# --- Save submission ---
sub = pd.DataFrame({
    "row_id": test.row_id.values,
    "Category:Misconception": joined_preds
})
#sub.to_csv("submission.csv", index=False)
print(sub.head())

prob_data = []
for i in range(len(test)):
    prob_dict = {f"prob_{j}": probs[i, top3[i, j]] for j in range(25)}  # Top 25
    prob_dict['row_id'] = test.row_id.values[i]
    prob_dict['top_classes'] = " ".join(top3_labels[i, :25])  # Top 25 class names
    prob_data.append(prob_dict)

prob_df = pd.DataFrame(prob_data)
prob_df.to_csv("submission_qwen3_4b_prob.csv", index=False)

In [ ]:
!python qwen3_4b_inference.py

In [ ]:
#自训练模型ver131
%%writefile qwen3_8b_inference.py
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, ModernBertForSequenceClassification, DataCollatorWithPadding
from sklearn.model_selection import train_test_split
from datasets import Dataset

from scipy.special import softmax

train               = pd.read_csv('/kaggle/input/map-charting-student-math-misunderstandings/train.csv')
test                = pd.read_csv('/kaggle/input/map-charting-student-math-misunderstandings/test.csv')
#test['row_id'] = range(len(test))


le                  = LabelEncoder()
train.Misconception     = train.Misconception.fillna('NA')
train['target']   = train.Category + ':' +train.Misconception
train['label']    = le.fit_transform(train['target'])

n_classes = len(le.classes_)
print(f"Train shape: {train.shape} with {n_classes} target classes")

idx = train.apply(lambda row: row.Category.split('_')[0],axis=1)=='True'
correct = train.loc[idx].copy()
correct['c'] = correct.groupby(['QuestionId','MC_Answer']).MC_Answer.transform('count')
correct = correct.sort_values('c',ascending=False)
correct = correct.drop_duplicates(['QuestionId'])
correct = correct[['QuestionId','MC_Answer']]
correct['is_correct'] = 1

train = train.merge(correct, on=['QuestionId','MC_Answer'], how='left')
train.is_correct = train.is_correct.fillna(0)

test = test.merge(correct, on=['QuestionId','MC_Answer'], how='left')
test.is_correct = test.is_correct.fillna(0)

def format_input(row):
    return (
            "Analyze the student's reasoning for the math problem below.\n"
            "### CONTEXT ###\n"
            f"Question: {row['QuestionText']}\n"
            f"Student's Answer: {row['MC_Answer']} ({'Correct' if row['is_correct'] else 'Incorrect'})\n"
            "### STUDENT EXPLANATION ###\n"
            f"{row['StudentExplanation']}\n"
            "### IDENTIFIED MISCONCEPTION ###\n"
    )

test['text'] = test.apply(format_input,axis=1)
ds_test = Dataset.from_pandas(test)


model_path = "/kaggle/input/map-qwen3-8b-quantize-merge/merged-exp131/"

tokenizer = AutoTokenizer.from_pretrained(model_path)


def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

ds_test = ds_test.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=n_classes,
    torch_dtype=torch.float16,
    device_map="auto")
model.config.pad_token_id = tokenizer.pad_token_id

#from peft import PeftModel
#model = PeftModel.from_pretrained(model, model_path)
#model.eval();

test_args = TrainingArguments(
    output_dir="./",
    do_train=False,
    do_predict=True,
    per_device_eval_batch_size=2, # Adjust as needed
    bf16=False, # TRAIN WITH BF16 IF LOCAL GPU IS NEWER GPU          
    fp16=True,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=test_args,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer)
)

predictions = trainer.predict(ds_test)
probs = softmax(predictions.predictions, axis=1)
predictions.predictions[:5]

top3           = np.argsort(-predictions.predictions, axis=1)
flat_top3      = top3.flatten()
decoded_labels = le.inverse_transform(flat_top3)
top3_labels_cat    = decoded_labels.reshape(top3.shape)
top3_labels_cat

joined_preds = []

for preds in top3_labels_cat:
    joined_preds.append(" ".join(preds))



# Save submission
sub = pd.DataFrame({
    "row_id": test.row_id.values,
    "Category:Misconception": joined_preds
})
#sub.to_csv("submission.csv", index=False)
sub.head()

prob_data = []
for i in range(len(test)):
    prob_dict = {f"prob_{j}": probs[i, top3[i, j]] for j in range(25)}  # Top 25
    prob_dict['row_id'] = test.row_id.values[i]
    prob_dict['top_classes'] = " ".join(top3_labels_cat[i, :25])  # Top 25 class names
    prob_data.append(prob_dict)

prob_df = pd.DataFrame(prob_data)
prob_df.to_csv("submission_qwen3_8b_prob.csv", index=False)

In [ ]:
!python qwen3_8b_inference.py

In [ ]:
#自训练模型ver166
%%writefile qwen3_8b_xml_inference.py
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, ModernBertForSequenceClassification, DataCollatorWithPadding
from sklearn.model_selection import train_test_split
from datasets import Dataset

from scipy.special import softmax

train               = pd.read_csv('/kaggle/input/map-charting-student-math-misunderstandings/train.csv')
test                = pd.read_csv('/kaggle/input/map-charting-student-math-misunderstandings/test.csv')
#test['row_id'] = range(len(test))


le                  = LabelEncoder()
train.Misconception     = train.Misconception.fillna('NA')
train['target']   = train.Category + ':' +train.Misconception
train['label']    = le.fit_transform(train['target'])

n_classes = len(le.classes_)
print(f"Train shape: {train.shape} with {n_classes} target classes")

idx = train.apply(lambda row: row.Category.split('_')[0],axis=1)=='True'
correct = train.loc[idx].copy()
correct['c'] = correct.groupby(['QuestionId','MC_Answer']).MC_Answer.transform('count')
correct = correct.sort_values('c',ascending=False)
correct = correct.drop_duplicates(['QuestionId'])
correct = correct[['QuestionId','MC_Answer']]
correct['is_correct'] = 1

train = train.merge(correct, on=['QuestionId','MC_Answer'], how='left')
train.is_correct = train.is_correct.fillna(0)

test = test.merge(correct, on=['QuestionId','MC_Answer'], how='left')
test.is_correct = test.is_correct.fillna(0)

def format_input_xml(row):
    correct_tag = "<is_correct>Yes</is_correct>" if row['is_correct'] else "<is_correct>No</is_correct>"
    
    prompt = (
        "<task_description>\n"
        "  Analyze the student's explanation for the following math problem and classify their misconception.\n"
        "</task_description>\n\n"
        "<problem_context>\n"
        "  <question_text>{question}</question_text>\n"
        "  <student_answer>{answer}</student_answer>\n"
        "  {correct_tag}\n"
        "</problem_context>\n\n"
        "<student_reasoning>\n"
        "  {explanation}\n"
        "</student_reasoning>\n\n"
        "<misconception_classification>"
    ).format(
        question=row['QuestionText'],
        answer=row['MC_Answer'],
        correct_tag=correct_tag,
        explanation=row['StudentExplanation']
    )
    return prompt

test['text'] = test.apply(format_input_xml,axis=1)
ds_test = Dataset.from_pandas(test)


model_path = "/kaggle/input/map-qwen3-8b-quantize-merge2/qwen-8b-merged-exp166/"

tokenizer = AutoTokenizer.from_pretrained(model_path)


def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

ds_test = ds_test.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=n_classes,
    torch_dtype=torch.float16,
    device_map="auto")
model.config.pad_token_id = tokenizer.pad_token_id

#from peft import PeftModel
#model = PeftModel.from_pretrained(model, model_path)
#model.eval();

test_args = TrainingArguments(
    output_dir="./",
    do_train=False,
    do_predict=True,
    per_device_eval_batch_size=2, # Adjust as needed
    bf16=False, # TRAIN WITH BF16 IF LOCAL GPU IS NEWER GPU          
    fp16=True,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=test_args,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer)
)

predictions = trainer.predict(ds_test)
probs = softmax(predictions.predictions, axis=1)
predictions.predictions[:5]

top3           = np.argsort(-predictions.predictions, axis=1)
flat_top3      = top3.flatten()
decoded_labels = le.inverse_transform(flat_top3)
top3_labels_cat    = decoded_labels.reshape(top3.shape)
top3_labels_cat

joined_preds = []

for preds in top3_labels_cat:
    joined_preds.append(" ".join(preds))



# Save submission
sub = pd.DataFrame({
    "row_id": test.row_id.values,
    "Category:Misconception": joined_preds
})
#sub.to_csv("submission.csv", index=False)
sub.head()

prob_data = []
for i in range(len(test)):
    prob_dict = {f"prob_{j}": probs[i, top3[i, j]] for j in range(25)}  # Top 25
    prob_dict['row_id'] = test.row_id.values[i]
    prob_dict['top_classes'] = " ".join(top3_labels_cat[i, :25])  # Top 25 class names
    prob_data.append(prob_dict)

prob_df = pd.DataFrame(prob_data)
prob_df.to_csv("qwen3_8b_xml_prob.csv", index=False)

In [ ]:
!python qwen3_8b_xml_inference.py

In [ ]:
#公开模型2-https://www.kaggle.com/datasets/jjsdfd22/deepseek-prover-v2-7b-bf16/versions/1
#https://www.kaggle.com/datasets/jjsdfd22/deepseek-prover-v2-7b-map/versions/1
%%writefile deepseek_prover_v2_7b_inference.py
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

import pandas as pd
import torch
import numpy as np
from argparse import Namespace
from sklearn.preprocessing import LabelEncoder

class DataProcessor:
    def __init__(self, args):
        self.args = args
        self.le = None
        self.isPreprocess = False
        self.correct_lookup = None

    def load_data(self):
        self.train_df = pd.read_csv(self.args.train_path)
        self.test_df = pd.read_csv(self.args.test_path)
        if self.args.use_extra_data:
            self.extra_df = pd.read_csv(self.args.extra_data_path)
            self.train_df = pd.concat([self.train_df, self.extra_df], ignore_index=True)

    def get_num_classes(self):
        if self.isPreprocess == False:
            return "please preprocess first"
        num_class = self.train_df['label'].nunique()
        return num_class

    def get_label_encoder(self):
        if self.le is None:
            raise ValueError("LabelEncoder not initialized. Please run preprocess first.")
        return self.le

    @staticmethod
    def format_input(row):
        correct_text = "Yes" if row['IsCorrect'] else "No"
        return (
            f"Question: {row['QuestionText']}\n"
            f"Answer: {row['MC_Answer']}\n"
            f"Correct? {correct_text}\n"
            f"Student Explanation: {row['StudentExplanation']}\n"
        )

    def preprocess(self):
        self.load_data()
        self.train_df['Misconception'] = self.train_df['Misconception'].fillna('NA')
        self.train_df['target'] = self.train_df['Category'] + ':' + self.train_df['Misconception']

        correct_samples = self.train_df[self.train_df['Category'].str.startswith('True', na=False)].copy()
        correct_samples['count'] = correct_samples.groupby(['QuestionId', 'MC_Answer'])['MC_Answer'].transform('count')
        most_popular_correct = correct_samples.sort_values('count', ascending=False).drop_duplicates(['QuestionId'])
        self.correct_lookup = most_popular_correct[['QuestionId', 'MC_Answer']].copy()
        self.correct_lookup['IsCorrect_flag'] = True

        self.train_df = self.train_df.merge(self.correct_lookup, on=['QuestionId', 'MC_Answer'], how='left')
        self.train_df['IsCorrect'] = self.train_df['IsCorrect_flag'].notna()
        self.train_df = self.train_df.drop(columns=['IsCorrect_flag'])

        self.le = LabelEncoder()
        self.train_df['label'] = self.le.fit_transform(self.train_df['target'])
        self.train_df['text'] = self.train_df.apply(self.format_input, axis=1)

        self.isPreprocess = True
        return self.train_df

    def inference_processor(self):
        if self.isPreprocess == False:
            return "Have you do the train? please preprocess first"
        self.test_df = self.test_df.merge(self.correct_lookup, on=['QuestionId', 'MC_Answer'], how='left')
        self.test_df['IsCorrect'] = self.test_df['IsCorrect_flag'].notna()
        self.test_df = self.test_df.drop(columns=['IsCorrect_flag'])
        self.test_df['text'] = self.test_df.apply(self.format_input, axis=1)
        return self.test_df

# inference
args = Namespace(
    train_path='/kaggle/input/map-charting-student-math-misunderstandings/train.csv',
    test_path='/kaggle/input/map-charting-student-math-misunderstandings/test.csv',
    use_extra_data=False,
    extra_data_path='datas.csv',
    model_dir="/kaggle/input/deepseek-prover-v2-7b-map",      
    inference_model_dir="/kaggle/input/deepseek-prover-v2-7b-map",
    mode='inference',
    model_name="/kaggle/input/deepseek-prover-v2-7b-bf16"   
)

DP = DataProcessor(args)
_ = DP.preprocess()

# %% 载入分词器与模型（仅推理）
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import PeftModel

# 分词器**从推理目录**载入，确保与训练时一致
tokenizer = AutoTokenizer.from_pretrained(args.inference_model_dir, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 基座 + LoRA 适配器（推理）
print(f"模式: 推理. 正在从 '{args.inference_model_dir}' 加载已训练的模型...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    args.model_name,
    num_labels=DP.get_num_classes(),
    device_map="auto",
    torch_dtype=torch.float16
)
model = PeftModel.from_pretrained(base_model, args.inference_model_dir)
model.config.pad_token_id = tokenizer.pad_token_id

MAX_LEN = 256
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LEN)

# %% 只处理测试集并推理
test_df = DP.inference_processor()
ds_test = Dataset.from_pandas(test_df[['text']])
ds_test = ds_test.map(tokenize_function, batched=True)

inference_args = TrainingArguments(
    do_train=False,
    do_eval=True,
    output_dir="./results_infer",
    per_device_eval_batch_size=16,
    fp16=True,
    bf16=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=inference_args,
    processing_class=tokenizer,
)

pred = trainer.predict(ds_test)
probs = torch.nn.functional.softmax(torch.tensor(pred.predictions), dim=1).numpy()

# %% 生成 submission.csv
top3 = np.argsort(-probs, axis=1)[:, :3]
flat_top3 = top3.flatten()
le = DP.get_label_encoder()
decoded_labels = le.inverse_transform(flat_top3)
top3_labels = decoded_labels.reshape(top3.shape)
joined_preds = [" ".join(row) for row in top3_labels]

sub = pd.DataFrame({
    "row_id": test_df.row_id.values,
    "Category:Misconception": joined_preds
})
#sub.to_csv("submission.csv", index=False)
print(sub.head())

top_indices = np.argsort(-probs, axis=1)
flat_indices = top_indices.flatten()
decoded_labels = le.inverse_transform(flat_indices)
top_labels = decoded_labels.reshape(top_indices.shape)
prob_data = []
for i in range(len(test_df)):
    prob_dict = {f"prob_{j}": probs[i, top_indices[i, j]] for j in range(25)}  # Top 25
    prob_dict['row_id'] = test_df.row_id.values[i]
    prob_dict['top_classes'] = " ".join(top_labels[i, :25])  # Top 25 class names
    prob_data.append(prob_dict)

prob_df = pd.DataFrame(prob_data)
prob_df.to_csv("submission_prover_7b_pub_prob.csv", index=False)

In [ ]:
!python deepseek_prover_v2_7b_inference.py

In [ ]:
#自训练模型ver171
%%writefile prover_fl_inference.py
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, ModernBertForSequenceClassification, DataCollatorWithPadding
from sklearn.model_selection import train_test_split
from datasets import Dataset
from scipy.special import softmax
train               = pd.read_csv('/kaggle/input/map-charting-student-math-misunderstandings/train.csv')
test                = pd.read_csv('/kaggle/input/map-charting-student-math-misunderstandings/test.csv')
#test['row_id'] = range(len(test))


le                  = LabelEncoder()
train.Misconception     = train.Misconception.fillna('NA')
train['target']   = train.Category + ':' +train.Misconception
train['label']    = le.fit_transform(train['target'])

n_classes = len(le.classes_)
print(f"Train shape: {train.shape} with {n_classes} target classes")

idx = train.apply(lambda row: row.Category.split('_')[0],axis=1)=='True'
correct = train.loc[idx].copy()
correct['c'] = correct.groupby(['QuestionId','MC_Answer']).MC_Answer.transform('count')
correct = correct.sort_values('c',ascending=False)
correct = correct.drop_duplicates(['QuestionId'])
correct = correct[['QuestionId','MC_Answer']]
correct['is_correct'] = 1

train = train.merge(correct, on=['QuestionId','MC_Answer'], how='left')
train.is_correct = train.is_correct.fillna(0)

test = test.merge(correct, on=['QuestionId','MC_Answer'], how='left')
test.is_correct = test.is_correct.fillna(0)

def format_input(row):
    return (
        "<task>Analyze the student's explanation for the following math problem and classify their misconception.</task>\n"
        f"<question>{row['QuestionText']}</question>\n"
        f"<student_answer>{row['MC_Answer']}</student_answer>\n"
        f"<is_correct>{'yes' if row['is_correct'] else 'no'}</is_correct>\n"
        f"<student_reasoning>{row['StudentExplanation']}</student_reasoning>\n"
        "<misconception>"
    )

test['text'] = test.apply(format_input,axis=1)
ds_test = Dataset.from_pandas(test)


model_path = "/kaggle/input/map-prover-quantize-merge/qwen-8b-merged-exp171/"

tokenizer = AutoTokenizer.from_pretrained(model_path)


def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

ds_test = ds_test.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=n_classes,
    torch_dtype=torch.float16,
    device_map="auto",)
model.config.pad_token_id = tokenizer.pad_token_id

#from peft import PeftModel
#model = PeftModel.from_pretrained(model, model_path)
#model.eval();

test_args = TrainingArguments(
    output_dir="./",
    do_train=False,
    do_predict=True,
    per_device_eval_batch_size=2, # Adjust as needed
    bf16=False, # TRAIN WITH BF16 IF LOCAL GPU IS NEWER GPU          
    fp16=True,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=test_args,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer)
)

predictions = trainer.predict(ds_test)
probs = softmax(predictions.predictions, axis=1)
predictions.predictions[:5]

top3           = np.argsort(-predictions.predictions, axis=1)
flat_top3      = top3.flatten()
decoded_labels = le.inverse_transform(flat_top3)
top3_labels_cat    = decoded_labels.reshape(top3.shape)
top3_labels_cat

joined_preds = []

for preds in top3_labels_cat:
    joined_preds.append(" ".join(preds))



# Save submission
sub = pd.DataFrame({
    "row_id": test.row_id.values,
    "Category:Misconception": joined_preds
})
#sub.to_csv("submission.csv", index=False)
sub.head()

prob_data = []
for i in range(len(test)):
    prob_dict = {f"prob_{j}": probs[i, top3[i, j]] for j in range(25)}  # Top 25
    prob_dict['row_id'] = test.row_id.values[i]
    prob_dict['top_classes'] = " ".join(top3_labels_cat[i, :25])  # Top 25 class names
    prob_data.append(prob_dict)

prob_df = pd.DataFrame(prob_data)
prob_df.to_csv("prover_fl_prob.csv", index=False)

In [ ]:
!python prover_fl_inference.py

In [ ]:
#公开模型3-lora权重-https://www.kaggle.com/datasets/jjsdfd22/qwen2-5-math-7b-instruct-map
#Qwen2.5-Math-7B-Instruct https://www.kaggle.com/models/qwen-lm/qwen2.5-math/Transformers/7b-instruct/1
%%writefile qwen2_5_math_7b_inference.py
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
import pandas as pd
import torch
import numpy as np
from argparse import Namespace
from sklearn.preprocessing import LabelEncoder

class DataProcessor:
    def __init__(self, args):
        self.args = args
        self.le = None
        self.isPreprocess = False
        self.correct_lookup = None

    def load_data(self):
        self.train_df = pd.read_csv(self.args.train_path)
        self.test_df = pd.read_csv(self.args.test_path)
        if self.args.use_extra_data:
            self.extra_df = pd.read_csv(self.args.extra_data_path)
            self.train_df = pd.concat([self.train_df, self.extra_df], ignore_index=True)

    def get_num_classes(self):
        if self.isPreprocess == False:
            return "please preprocess first"
        num_class = self.train_df['label'].nunique()
        return num_class

    def get_label_encoder(self):
        if self.le is None:
            raise ValueError("LabelEncoder not initialized. Please run preprocess first.")
        return self.le

    @staticmethod
    def format_input(row):
        correct_text = "Yes" if row['IsCorrect'] else "No"
        return (
            f"Question: {row['QuestionText']}\n"
            f"Answer: {row['MC_Answer']}\n"
            f"Correct? {correct_text}\n"
            f"Student Explanation: {row['StudentExplanation']}\n"
        )

    def preprocess(self):
        self.load_data()
        self.train_df['Misconception'] = self.train_df['Misconception'].fillna('NA')
        self.train_df['target'] = self.train_df['Category'] + ':' + self.train_df['Misconception']

        correct_samples = self.train_df[self.train_df['Category'].str.startswith('True', na=False)].copy()
        correct_samples['count'] = correct_samples.groupby(['QuestionId', 'MC_Answer'])['MC_Answer'].transform('count')
        most_popular_correct = correct_samples.sort_values('count', ascending=False).drop_duplicates(['QuestionId'])
        self.correct_lookup = most_popular_correct[['QuestionId', 'MC_Answer']].copy()
        self.correct_lookup['IsCorrect_flag'] = True
        self.train_df = self.train_df.merge(self.correct_lookup, on=['QuestionId', 'MC_Answer'], how='left')
        self.train_df['IsCorrect'] = self.train_df['IsCorrect_flag'].notna()
        self.train_df = self.train_df.drop(columns=['IsCorrect_flag'])
        self.le = LabelEncoder()
        self.train_df['label'] = self.le.fit_transform(self.train_df['target'])
        self.train_df['text'] = self.train_df.apply(self.format_input, axis=1)

        self.isPreprocess = True
        return self.train_df

    def inference_processor(self):
        if self.isPreprocess == False:
            return "Have you do the train? please preprocess first"
        self.test_df = self.test_df.merge(self.correct_lookup, on=['QuestionId', 'MC_Answer'], how='left')
        self.test_df['IsCorrect'] = self.test_df['IsCorrect_flag'].notna()
        self.test_df = self.test_df.drop(columns=['IsCorrect_flag'])
        self.test_df['text'] = self.test_df.apply(self.format_input, axis=1)
        return self.test_df


args = Namespace(
    train_path='/kaggle/input/map-charting-student-math-misunderstandings/train.csv',
    test_path='/kaggle/input/map-charting-student-math-misunderstandings/test.csv',
    use_extra_data=False,
    extra_data_path='NO_DATA.csv',
    model_dir="/kaggle/input/qwen2-5-math-7b-instruct-map",      
    inference_model_dir="/kaggle/input/qwen2-5-math-7b-instruct-map",
    mode='inference',
    model_name="/kaggle/input/qwen2.5-math/transformers/7b-instruct/1"   
)

DP = DataProcessor(args)
_ = DP.preprocess()

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import PeftModel

# 分词器载入
tokenizer = AutoTokenizer.from_pretrained(args.inference_model_dir, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 基座 + LoRA
print(f"模式: 推理. 正在从 '{args.inference_model_dir}' 加载已训练的模型...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    args.model_name,
    num_labels=DP.get_num_classes(),
    device_map="auto",
    torch_dtype=torch.float16
)
model = PeftModel.from_pretrained(base_model, args.inference_model_dir)
model.config.pad_token_id = tokenizer.pad_token_id

MAX_LEN = 256
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LEN)

test_df = DP.inference_processor()
ds_test = Dataset.from_pandas(test_df[['text']])
ds_test = ds_test.map(tokenize_function, batched=True)

inference_args = TrainingArguments(
    do_train=False,
    do_eval=True,
    output_dir="./results_infer",
    per_device_eval_batch_size=16,
    fp16=True,
    bf16=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=inference_args,
    processing_class=tokenizer,
)

pred = trainer.predict(ds_test)
probs = torch.nn.functional.softmax(torch.tensor(pred.predictions), dim=1).numpy()

top3 = np.argsort(-probs, axis=1)[:, :3]
flat_top3 = top3.flatten()
le = DP.get_label_encoder()
decoded_labels = le.inverse_transform(flat_top3)
top3_labels = decoded_labels.reshape(top3.shape)
joined_preds = [" ".join(row) for row in top3_labels]

sub = pd.DataFrame({
    "row_id": test_df.row_id.values,
    "Category:Misconception": joined_preds
})
#sub.to_csv("submission.csv", index=False)
print(sub.head())

top_indices = np.argsort(-probs, axis=1)
flat_indices = top_indices.flatten()
decoded_labels = le.inverse_transform(flat_indices)
top_labels = decoded_labels.reshape(top_indices.shape)
prob_data = []
for i in range(len(test_df)):
    prob_dict = {f"prob_{j}": probs[i, top_indices[i, j]] for j in range(25)}  # Top 25
    prob_dict['row_id'] = test_df.row_id.values[i]
    prob_dict['top_classes'] = " ".join(top_labels[i, :25])  # Top 25 class names
    prob_data.append(prob_dict)

prob_df = pd.DataFrame(prob_data)
prob_df.to_csv("submission_qwen2_5_math_7b_pub_prob.csv", index=False)

In [ ]:
!python qwen2_5_math_7b_inference.py

In [ ]:
!pip install --upgrade --no-index --find-links=/kaggle/input/transformers-4-56-1-and-deps transformers -qq

In [ ]:
#公开模型4-lora权重https://www.kaggle.com/datasets/jjsdfd22/hunyuan-7b-instruct-bf16/versions/1
#hunyuan-7b-instruct量化版-https://www.kaggle.com/datasets/jjsdfd22/hunyuan-7b-instruct-bf16/versions/1
%%writefile hunyuan7b_pub_inference.py
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
import pandas as pd
import torch
import numpy as np
from argparse import Namespace
from sklearn.preprocessing import LabelEncoder

class DataProcessor:
    def __init__(self, args):
        self.args = args
        self.le = None
        self.isPreprocess = False
        self.correct_lookup = None

    def load_data(self):
        self.train_df = pd.read_csv(self.args.train_path)
        self.test_df = pd.read_csv(self.args.test_path)
        if self.args.use_extra_data:
            self.extra_df = pd.read_csv(self.args.extra_data_path)
            self.train_df = pd.concat([self.train_df, self.extra_df], ignore_index=True)

    def get_num_classes(self):
        if self.isPreprocess == False:
            return "please preprocess first"
        num_class = self.train_df['label'].nunique()
        return num_class

    def get_label_encoder(self):
        if self.le is None:
            raise ValueError("LabelEncoder not initialized. Please run preprocess first.")
        return self.le

    @staticmethod
    def format_input(row):
        correct_text = "Yes" if row['IsCorrect'] else "No"
        return (
            f"Question: {row['QuestionText']}\n"
            f"Answer: {row['MC_Answer']}\n"
            f"Correct? {correct_text}\n"
            f"Student Explanation: {row['StudentExplanation']}\n"
        )

    def preprocess(self):
        self.load_data()
        self.train_df['Misconception'] = self.train_df['Misconception'].fillna('NA')
        self.train_df['target'] = self.train_df['Category'] + ':' + self.train_df['Misconception']

        correct_samples = self.train_df[self.train_df['Category'].str.startswith('True', na=False)].copy()
        correct_samples['count'] = correct_samples.groupby(['QuestionId', 'MC_Answer'])['MC_Answer'].transform('count')
        most_popular_correct = correct_samples.sort_values('count', ascending=False).drop_duplicates(['QuestionId'])
        self.correct_lookup = most_popular_correct[['QuestionId', 'MC_Answer']].copy()
        self.correct_lookup['IsCorrect_flag'] = True

        self.train_df = self.train_df.merge(self.correct_lookup, on=['QuestionId', 'MC_Answer'], how='left')
        self.train_df['IsCorrect'] = self.train_df['IsCorrect_flag'].notna()
        self.train_df = self.train_df.drop(columns=['IsCorrect_flag'])

        self.le = LabelEncoder()
        self.train_df['label'] = self.le.fit_transform(self.train_df['target'])
        self.train_df['text'] = self.train_df.apply(self.format_input, axis=1)

        self.isPreprocess = True
        return self.train_df

    def inference_processor(self):
        if self.isPreprocess == False:
            return "Have you do the train? please preprocess first"
        self.test_df = self.test_df.merge(self.correct_lookup, on=['QuestionId', 'MC_Answer'], how='left')
        self.test_df['IsCorrect'] = self.test_df['IsCorrect_flag'].notna()
        self.test_df = self.test_df.drop(columns=['IsCorrect_flag'])
        self.test_df['text'] = self.test_df.apply(self.format_input, axis=1)
        return self.test_df

# inference
args = Namespace(
    train_path='/kaggle/input/map-charting-student-math-misunderstandings/train.csv',
    test_path='/kaggle/input/map-charting-student-math-misunderstandings/test.csv',
    use_extra_data=False,
    extra_data_path='no_datas.csv',
    model_dir="/kaggle/input/hunyuan-7b-instruct-map",      
    inference_model_dir="/kaggle/input/hunyuan-7b-instruct-map",
    mode='inference',
    model_name="/kaggle/input/hunyuan-7b-instruct-bf16"   
)

DP = DataProcessor(args)
_ = DP.preprocess()

#载入分词器与模型
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(args.inference_model_dir, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 基座 + LoRA
print(f"模式: 推理. 正在从 '{args.inference_model_dir}' 加载已训练的模型...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    args.model_name,
    num_labels=DP.get_num_classes(),
    device_map="auto",
    torch_dtype=torch.float16
)
model = PeftModel.from_pretrained(base_model, args.inference_model_dir)
model.config.pad_token_id = tokenizer.pad_token_id

MAX_LEN = 256
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LEN)

# 只处理测试集并推理
test_df = DP.inference_processor()
ds_test = Dataset.from_pandas(test_df[['text']])
ds_test = ds_test.map(tokenize_function, batched=True)

inference_args = TrainingArguments(
    do_train=False,
    do_eval=True,
    output_dir="./results_infer",
    per_device_eval_batch_size=16,
    fp16=True,
    bf16=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=inference_args,
    processing_class=tokenizer,
)

pred = trainer.predict(ds_test)
probs = torch.nn.functional.softmax(torch.tensor(pred.predictions), dim=1).numpy()

# Result
top3 = np.argsort(-probs, axis=1)[:, :3]
flat_top3 = top3.flatten()
le = DP.get_label_encoder()
decoded_labels = le.inverse_transform(flat_top3)
top3_labels = decoded_labels.reshape(top3.shape)
joined_preds = [" ".join(row) for row in top3_labels]

sub = pd.DataFrame({
    "row_id": test_df.row_id.values,
    "Category:Misconception": joined_preds
})
#sub.to_csv("submission.csv", index=False)
sub.head()

top_indices = np.argsort(-probs, axis=1)
flat_indices = top_indices.flatten()
decoded_labels = le.inverse_transform(flat_indices)
top_labels = decoded_labels.reshape(top_indices.shape)
prob_data = []
for i in range(len(test_df)):
    prob_dict = {f"prob_{j}": probs[i, top_indices[i, j]] for j in range(25)}  # Top 25
    prob_dict['row_id'] = test_df.row_id.values[i]
    prob_dict['top_classes'] = " ".join(top_labels[i, :25])  # Top 25 class names
    prob_data.append(prob_dict)

prob_df = pd.DataFrame(prob_data)
prob_df.to_csv("submission_hunyuan_7b_pub_prob.csv", index=False)

In [ ]:
!python hunyuan7b_pub_inference.py

In [ ]:
#自训练模型ver142
%%writefile hunyuan7b_fl_inference.py
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, ModernBertForSequenceClassification, DataCollatorWithPadding
from sklearn.model_selection import train_test_split
from datasets import Dataset
from scipy.special import softmax
train               = pd.read_csv('/kaggle/input/map-charting-student-math-misunderstandings/train.csv')
test                = pd.read_csv('/kaggle/input/map-charting-student-math-misunderstandings/test.csv')
#test['row_id'] = range(len(test))


le                  = LabelEncoder()
train.Misconception     = train.Misconception.fillna('NA')
train['target']   = train.Category + ':' +train.Misconception
train['label']    = le.fit_transform(train['target'])

n_classes = len(le.classes_)
print(f"Train shape: {train.shape} with {n_classes} target classes")

idx = train.apply(lambda row: row.Category.split('_')[0],axis=1)=='True'
correct = train.loc[idx].copy()
correct['c'] = correct.groupby(['QuestionId','MC_Answer']).MC_Answer.transform('count')
correct = correct.sort_values('c',ascending=False)
correct = correct.drop_duplicates(['QuestionId'])
correct = correct[['QuestionId','MC_Answer']]
correct['is_correct'] = 1

train = train.merge(correct, on=['QuestionId','MC_Answer'], how='left')
train.is_correct = train.is_correct.fillna(0)

test = test.merge(correct, on=['QuestionId','MC_Answer'], how='left')
test.is_correct = test.is_correct.fillna(0)

def format_input(row):
    return (
            "Analyze the student's reasoning for the math problem below.\n"
            "### CONTEXT ###\n"
            f"Question: {row['QuestionText']}\n"
            f"Student's Answer: {row['MC_Answer']} ({'Correct' if row['is_correct'] else 'Incorrect'})\n"
            "### STUDENT EXPLANATION ###\n"
            f"{row['StudentExplanation']}\n"
            "### IDENTIFIED MISCONCEPTION ###\n"
    )

test['text'] = test.apply(format_input,axis=1)
ds_test = Dataset.from_pandas(test)


model_path = "/kaggle/input/map-hunyuan7b-quantize-merge/hunyuan-7b-merged-exp142/"

tokenizer = AutoTokenizer.from_pretrained(model_path)


def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

ds_test = ds_test.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=n_classes,
    torch_dtype=torch.float16,
    device_map="auto",)
model.config.pad_token_id = tokenizer.pad_token_id

#from peft import PeftModel
#model = PeftModel.from_pretrained(model, model_path)
#model.eval();

test_args = TrainingArguments(
    output_dir="./",
    do_train=False,
    do_predict=True,
    per_device_eval_batch_size=2, # Adjust as needed
    bf16=False, # TRAIN WITH BF16 IF LOCAL GPU IS NEWER GPU          
    fp16=True,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=test_args,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer)
)

predictions = trainer.predict(ds_test)
probs = softmax(predictions.predictions, axis=1)
predictions.predictions[:5]

top3           = np.argsort(-predictions.predictions, axis=1)
flat_top3      = top3.flatten()
decoded_labels = le.inverse_transform(flat_top3)
top3_labels_cat    = decoded_labels.reshape(top3.shape)
top3_labels_cat

joined_preds = []

for preds in top3_labels_cat:
    joined_preds.append(" ".join(preds))



# Save submission
sub = pd.DataFrame({
    "row_id": test.row_id.values,
    "Category:Misconception": joined_preds
})
#sub.to_csv("submission.csv", index=False)
sub.head()

prob_data = []
for i in range(len(test)):
    prob_dict = {f"prob_{j}": probs[i, top3[i, j]] for j in range(25)}  # Top 25
    prob_dict['row_id'] = test.row_id.values[i]
    prob_dict['top_classes'] = " ".join(top3_labels_cat[i, :25])  # Top 25 class names
    prob_data.append(prob_dict)

prob_df = pd.DataFrame(prob_data)
prob_df.to_csv("submission_hunyuan_7b_fl_prob.csv", index=False)

In [ ]:
!python hunyuan7b_fl_inference.py

In [ ]:
#公开模型5/6-https://www.kaggle.com/datasets/jaytonde/deekseepmath-7b-map-competition
#https://www.kaggle.com/datasets/jaytonde/qwen3-8b-map-competition
%%writefile qwen3_deepseek_inference.py

# we do parallel inference, for deepseek and qwen3
import os
import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
import threading
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
from scipy.special import softmax
from tqdm import tqdm
import time

os.environ["TOKENIZERS_PARALLELISM"] = "false"


train = pd.read_csv('/kaggle/input/map-charting-student-math-misunderstandings/train.csv')
test  = pd.read_csv('/kaggle/input/map-charting-student-math-misunderstandings/test.csv')

model_paths = [
    "/kaggle/input/deekseepmath-7b-map-competition/MAP_EXP_09_FULL",
   "/kaggle/input/qwen3-8b-map-competition/MAP_EXP_16_FULL"]

def format_input(row):
    x = "This answer is correct."
    if not row['is_correct']:
        x = "This is answer is incorrect."
    return (
        f"Question: {row['QuestionText']}\n"
        f"Answer: {row['MC_Answer']}\n"
        f"{x}\n"
        f"Student Explanation: {row['StudentExplanation']}")


le = LabelEncoder()
train.Misconception  = train.Misconception.fillna('NA')
train['target']   = train.Category + ':' +train.Misconception
train['label']    = le.fit_transform(train['target'])

n_classes = len(le.classes_)
print(f"Train shape: {train.shape} with {n_classes} target classes")
idx = train.apply(lambda row: row.Category.split('_')[0],axis=1)=='True'
correct = train.loc[idx].copy()
correct['c'] = correct.groupby(['QuestionId','MC_Answer']).MC_Answer.transform('count')
correct = correct.sort_values('c',ascending=False)
correct = correct.drop_duplicates(['QuestionId'])
correct = correct[['QuestionId','MC_Answer']]
correct['is_correct'] = 1

test = test.merge(correct, on=['QuestionId','MC_Answer'], how='left')
test.is_correct = test.is_correct.fillna(0)
test['text'] = test.apply(format_input,axis=1)
ds_test = Dataset.from_pandas(test)


def run_inference_on_gpu(model_path, gpu_id, test_data, output_name):
    """Run inference for one model on one GPU"""
    
    device = f"cuda:{gpu_id}"
    print(f"Loading {output_name} on {device}...")
    
    # Load model
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path, 
        device_map=device, 
        torch_dtype=torch.float16
    )
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.eval()
    
    # Tokenize function
    def tokenize(batch):
        return tokenizer(batch["text"], 
                        truncation=True,
                        max_length=256)
    
    ds_test = Dataset.from_pandas(test_data[['text']])
    ds_test = ds_test.map(tokenize, batched=True, remove_columns=['text'])
    
    # Data collator
    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        padding=True,
        return_tensors="pt"
    )
    
    # DataLoader
    dataloader = DataLoader(
        ds_test,
        batch_size=4,
        shuffle=False,
        collate_fn=data_collator,
        pin_memory=True,
        num_workers=0
    )
    
    # Inference
    all_logits = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"{output_name}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            all_logits.append(outputs.logits.float().cpu().numpy())
    
    predictions = np.concatenate(all_logits, axis=0)
    
    # Process results
    probs = softmax(predictions, axis=1)
    top_indices = np.argsort(-probs, axis=1)
    
    # Decode labels
    flat_indices = top_indices.flatten()
    decoded_labels = le.inverse_transform(flat_indices)
    top_labels = decoded_labels.reshape(top_indices.shape)
    
    # Save top-3 submission
    joined_preds = [" ".join(row[:3]) for row in top_labels]
    sub = pd.DataFrame({
        "row_id": test_data.row_id.values,
        "Category:Misconception": joined_preds
    })
    sub.to_csv(f"submission_{output_name}.csv", index=False)
    
    # Save probabilities for ensemble
    prob_data = []
    for i in range(len(predictions)):
        prob_dict = {f"prob_{j}": probs[i, top_indices[i, j]] for j in range(25)}
        prob_dict['row_id'] = test_data.row_id.values[i]
        prob_dict['top_classes'] = " ".join(top_labels[i, :25])
        prob_data.append(prob_dict)
    
    prob_df = pd.DataFrame(prob_data)
    prob_df.to_csv(f"submission_{output_name}_probabilities.csv", index=False)
    
    print(f" {output_name} completed - saved submission and probabilities")
    
    # Clean up GPU memory
    del model, tokenizer
    torch.cuda.empty_cache()

print(" Starting multi-GPU inference...")
start_time = time.time()

threads = []
gpu_assignments = [
    (model_paths[0], 0, "deepseek"),
    (model_paths[1], 1, "qwen3"),
]

# Start threads
for model_path, gpu_id, name in gpu_assignments:
    if gpu_id < torch.cuda.device_count():  
        thread = threading.Thread(
            target=run_inference_on_gpu,
            args=(model_path, gpu_id, test, name)
        )
        threads.append(thread)
        thread.start()
        time.sleep(10)  # Stagger starts to avoid memory issues

# Wait for completion
for thread in threads:
    thread.join()

end_time = time.time()
print(f" completed in {end_time - start_time:.2f} seconds!")

In [ ]:
import time 
#!python /kaggle/working/gemma2_inference.py
time.sleep(10)
!python /kaggle/working/qwen3_deepseek_inference.py

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
from scipy.special import softmax



def extract_class_probabilities(row, model_suffix='', top_k=25):
    """Extract class names and probabilities from a row"""
    # Get top classes
    classes_col = f'top_classes{model_suffix}'
    if classes_col in row:
        classes = row[classes_col].split(' ')[:top_k]
    else:
        return {}
    # Get probabilities
    class_probs = {}
    for i in range(min(top_k, len(classes))):
        prob_col = f'prob_{i}{model_suffix}'
        if prob_col in row:
            class_probs[classes[i]] = row[prob_col]
    return class_probs


def ensemble_with_disagreement_handling(prob_files, model_weights=None, top_k=3):
    n_models = len(prob_files)
    prob_dfs = []
    final_predictions = []
    
    for file_path in prob_files:
        df = pd.read_csv(file_path)
        prob_dfs.append(df)
    
    # Merge on row_id
    merged_df = prob_dfs[0]
    for i, df in enumerate(prob_dfs[1:], 1):
        merged_df = pd.merge(merged_df, df, on='row_id', suffixes=('', f'_model{i+1}'))
      
    for idx, row in merged_df.iterrows():
        
        # Extract probabilities from each model
        all_class_probs = []
        for i in range(n_models):
            suffix = f'_model{i+1}' if i > 0 else ''
            class_probs = extract_class_probabilities(row, suffix, top_k=25)
            all_class_probs.append(class_probs)
        
        # Get all unique classes
        all_classes = set()
        for class_probs in all_class_probs:
            all_classes.update(class_probs.keys())
        
        # Calculate agreement and disagreement
        class_votes = defaultdict(int)
        class_total_prob = defaultdict(float)
        class_max_prob = defaultdict(float)
        
        for i, class_probs in enumerate(all_class_probs):
            weight = model_weights[i]
            
            for class_name, prob in class_probs.items():
                class_votes[class_name] += 1
                class_total_prob[class_name] += prob * weight
                class_max_prob[class_name] = max(class_max_prob[class_name], prob * weight)
        
        final_scores = {}
        for class_name in all_classes:
            
            # Base score: weighted average probability
            base_score = class_total_prob[class_name]
            
            # Agreement : classes predicted by more models get boost
            agreement_bonus = class_votes[class_name] / n_models
            
            # Confidence bonus: classes with high max probability get boost
            confidence_bonus = class_max_prob[class_name]
            
            # Combined score
            final_scores[class_name] = (
                base_score * 0.6 +           # 60% base probs
                agreement_bonus * 0.3 +      # 30% agreement
                confidence_bonus * 0.1       # 10% confidence
            )
        
        # Sort and get top-k
        sorted_classes = sorted(final_scores.items(), key=lambda x: -x[1])
        top_classes = [class_name for class_name, _ in sorted_classes[:top_k]]
        
        final_predictions.append(' '.join(top_classes))
    
    return final_predictions


w1 = 1.0
w2 = 1.0
w3 = 0.7
w4 = 1.0
w5 = 1.0
w6 = 0.8
w7 = 1.0
w8 = 1.0
w9 = 1.0
w10 = 1.3
prob_files = [
    '/kaggle/working/submission_hunyuan_7b_pub_prob.csv',
    '/kaggle/working/submission_qwen3_4b_prob.csv',
    '/kaggle/working/submission_qwen3_8b_prob.csv',
    "/kaggle/working/submission_hunyuan_7b_fl_prob.csv",
    '/kaggle/working/submission_deepseek_probabilities.csv',
    '/kaggle/working/submission_qwen3_probabilities.csv',
    "submission_qwen2_5_math_7b_pub_prob.csv",
    "submission_prover_7b_pub_prob.csv",
    "prover_fl_prob.csv",
    "qwen3_8b_xml_prob.csv",

]


predictions = ensemble_with_disagreement_handling(
        prob_files, 
        model_weights=[w1, w2, w3, w4, w5,  w6, w7, w8, w9, w10],  
        top_k=3
    )
    
test_df = pd.read_csv('/kaggle/input/map-charting-student-math-misunderstandings/test.csv')

submission = pd.DataFrame({
    'row_id': test_df.row_id.values,
    'Category:Misconception': predictions
})

submission.to_csv('submission.csv', index=False)
submission